<a href="https://colab.research.google.com/github/dianoshdinparvar-ship-it/POWERMODE-ENGINE/blob/main/powermode_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================

# 🔥 POWERMODE OSLO v7.1 — DUAL ENGINE FULL ALPHA REGIME

# ============================================================

!pip install yfinance pandas numpy openpyxl -q

import yfinance as yf

import pandas as pd

import numpy as np

import warnings

from datetime import date

warnings.filterwarnings("ignore")

# ============================================================

# SETTINGS

# ============================================================

LOOKBACK_DAYS = 120

TRADE_HOLD_DAYS = 10

STRUCTURE_HOLD_DAYS = 40

MIN_GAP_DAYS = 5

TODAY = date.today().isoformat()

# ============================================================

# OSLO / OSE UNIVERSE

# ============================================================

tickers = [

    "EQNR.OL","DNB.OL","KOG.OL","NHY.OL","YAR.OL","STB.OL",

    "AKRBP.OL","VAR.OL","DNO.OL","PEN.OL","SUBC.OL","TGS.OL",

    "BWO.OL","OET.OL","AKSO.OL","BWE.OL","DOFG.OL","NORAM.OL","BNOR.OL",

    "BWLPG.OL","HAFNI.OL","FRO.OL","MPCC.OL","KCC.OL",

    "HAUTO.OL","HSHIP.OL","WAWI.OL","WWI.OL","WWIB.OL",

    "CADLR.OL","SOAG.OL","SMOP.OL",

    "SBNOR.OL","MING.OL","GJF.OL","PROT.OL","PARB.OL",

    "SPOL.OL","SKUE.OL","BONHR.OL",

    "KIT.OL","NOD.OL","KID.OL","TOM.OL","ELK.OL","AFG.OL",

    "VEI.OL","ATEA.OL","NORBT.OL","MULTI.OL",

    "MOWI.OL","SALM.OL","LSG.OL","AUSS.OL","AKH.OL",

    "ABL.OL","ENVIP.OL","ARCH.OL","AUTO.OL","ODL.OL",

    "SATS.OL","NAPA.OL","LINK.OL","DFENS.OL",

    "SCATC.OL","HEX.OL","MPCES.OL","TEL.OL"

]

tickers = sorted(list(set(tickers)))

bad_tickers = ["GOGL.OL","CRAYN.OL","XXL.OL","CMBTO.OL"]

tickers = [t for t in tickers if t not in bad_tickers]

BENCH = "OSEBX.OL"

# ============================================================

# INDICATORS

# ============================================================

def ema(x, n):

    return x.ewm(span=n, adjust=False).mean()

def dema(x, n):

    e1 = ema(x, n)

    e2 = ema(e1, n)

    return 2 * e1 - e2

def kama(series, er_period=10, fast=2, slow=30):

    change = abs(series - series.shift(er_period))

    vol = series.diff().abs().rolling(er_period).sum()

    er = change / (vol + 1e-9)

    fast_sc = 2 / (fast + 1)

    slow_sc = 2 / (slow + 1)

    sc = (er * (fast_sc - slow_sc) + slow_sc) ** 2

    out = pd.Series(index=series.index, dtype=float)

    out.iloc[0] = series.iloc[0]

    for i in range(1, len(series)):

        if np.isnan(sc.iloc[i]):

            out.iloc[i] = out.iloc[i - 1]

        else:

            out.iloc[i] = out.iloc[i - 1] + sc.iloc[i] * (series.iloc[i] - out.iloc[i - 1])

    return out

def rsi(x, n=14):

    d = x.diff()

    up = d.clip(lower=0)

    down = -d.clip(upper=0)

    rs = up.ewm(alpha=1/n, adjust=False).mean() / (

        down.ewm(alpha=1/n, adjust=False).mean() + 1e-9

    )

    return 100 - (100 / (1 + rs))

def macd_hist(x):

    macd = ema(x, 12) - ema(x, 26)

    sig = ema(macd, 9)

    return macd - sig

def dmi(df, n=10):

    high = df["High"]

    low = df["Low"]

    close = df["Close"]

    up_move = high.diff()

    down_move = -low.diff()

    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)

    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)

    tr1 = high - low

    tr2 = abs(high - close.shift())

    tr3 = abs(low - close.shift())

    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    atr = tr.ewm(alpha=1/n, adjust=False).mean()

    plus_di = (

        100 * pd.Series(plus_dm, index=df.index)

        .ewm(alpha=1/n, adjust=False)

        .mean() / (atr + 1e-9)

    )

    minus_di = (

        100 * pd.Series(minus_dm, index=df.index)

        .ewm(alpha=1/n, adjust=False)

        .mean() / (atr + 1e-9)

    )

    return plus_di, minus_di

# ============================================================

# CORE FEATURE ENGINE

# ============================================================

def add_core(df):

    df = df.copy()

    close = df["Close"]

    df["DEMA50"] = dema(close, 50)

    df["DEMA200"] = dema(close, 200)

    df["KAMA"] = kama(close)

    df["RSI"] = rsi(close)

    df["MACD_HIST"] = macd_hist(close)

    df["MACD_SLOPE"] = df["MACD_HIST"] - df["MACD_HIST"].shift(1)

    pdi, mdi = dmi(df)

    df["DI_PLUS"] = pdi

    df["DI_MINUS"] = mdi

    df["DMI_SPREAD"] = df["DI_PLUS"] - df["DI_MINUS"]

    df["DMI_ACCEL"] = df["DMI_SPREAD"] - df["DMI_SPREAD"].shift(1)

    df["RS20"] = ((close / close.shift(20)) - 1) * 100

    df["RS60"] = ((close / close.shift(60)) - 1) * 100

    df["Persistence"] = (df["Close"] > df["KAMA"]).rolling(20).sum()

    return df.replace([np.inf, -np.inf], np.nan).dropna()

# ============================================================

# ENGINE 1 — 10D TRADE ENGINE

# ============================================================

def trade_signal(core):

    row = core.iloc[-1]

    prev = core.iloc[-2]

    rsi_now = row["RSI"]

    macd_improving = row["MACD_HIST"] > prev["MACD_HIST"]

    dmi_positive = row["DI_PLUS"] > row["DI_MINUS"]

    rs20 = row["RS20"]

    if 52 <= rsi_now <= 68 and macd_improving and dmi_positive and -5 <= rs20 <= 15:

        return "10D TRADE"

    return "NO SIGNAL"

# ============================================================

# ENGINE 2 — 40D STRUCTURE ENGINE

# ============================================================

def structure_signal(core):

    row = core.iloc[-1]

    prev = core.iloc[-2]

    persistence = row["Persistence"]

    rsi_now = row["RSI"]

    macd_improving = row["MACD_HIST"] > prev["MACD_HIST"]

    dmi_positive = row["DI_PLUS"] > row["DI_MINUS"]

    rs20 = row["RS20"]

    if persistence >= 14 and 50 <= rsi_now <= 68 and macd_improving and dmi_positive and 0 <= rs20 <= 20:

        return "40D STRUCTURE"

    return "NO SIGNAL"

# ============================================================

# DOWNLOAD DATA

# ============================================================

print("Downloading data...")

raw = yf.download(

    tickers + [BENCH],

    period="3y",

    interval="1d",

    group_by="ticker",

    auto_adjust=True,

    progress=False,

    threads=False

)

data = {}

for t in tickers + [BENCH]:

    try:

        df = raw[t].dropna()

        if isinstance(df.columns, pd.MultiIndex):

            df.columns = df.columns.get_level_values(0)

        df = df[["Open","High","Low","Close","Volume"]].replace(

            [np.inf, -np.inf], np.nan

        ).dropna()

        if len(df) > 250:

            data[t] = df

    except:

        pass

print("Loaded:", len(data) - (1 if BENCH in data else 0))

missing = [t for t in tickers if t not in data]

print("\nMissing / not loaded:")

print(missing)

if BENCH not in data:

    raise ValueError("Benchmark OSEBX.OL not loaded. Try BENCH='^OSEBX' or check Yahoo ticker.")

bench = data[BENCH].copy()

# ============================================================

# EDGE TRACKER

# ============================================================

dates = bench.index[

    -(LOOKBACK_DAYS + STRUCTURE_HOLD_DAYS):

    -STRUCTURE_HOLD_DAYS

]

trade_rows = []

structure_rows = []

for d in dates:

    for t in tickers:

        try:

            df = data.get(t)

            if df is None:

                continue

            hist = df[df.index <= d]

            future = df[df.index > d]

            if len(hist) < 250 or len(future) < STRUCTURE_HOLD_DAYS:

                continue

            core = add_core(hist.tail(250))

            if len(core) < 5:

                continue

            t_signal = trade_signal(core)

            if t_signal == "10D TRADE":

                entry = hist["Close"].iloc[-1]

                exit_trade = future["Close"].iloc[TRADE_HOLD_DAYS - 1]

                ret_trade = (exit_trade / entry - 1) * 100

                trade_rows.append([

                    d, t,

                    round(core.iloc[-1]["RSI"], 1),

                    round(core.iloc[-1]["RS20"], 1),

                    round(core.iloc[-1]["RS60"], 1),

                    int(core.iloc[-1]["Persistence"]),

                    round(core.iloc[-1]["MACD_HIST"], 3),

                    round(core.iloc[-1]["MACD_SLOPE"], 3),

                    round(core.iloc[-1]["DI_PLUS"], 1),

                    round(core.iloc[-1]["DI_MINUS"], 1),

                    round(core.iloc[-1]["DMI_ACCEL"], 2),

                    round(ret_trade, 2)

                ])

            s_signal = structure_signal(core)

            if s_signal == "40D STRUCTURE":

                entry = hist["Close"].iloc[-1]

                exit_structure = future["Close"].iloc[STRUCTURE_HOLD_DAYS - 1]

                ret_structure = (exit_structure / entry - 1) * 100

                structure_rows.append([

                    d, t,

                    round(core.iloc[-1]["RSI"], 1),

                    round(core.iloc[-1]["RS20"], 1),

                    round(core.iloc[-1]["RS60"], 1),

                    int(core.iloc[-1]["Persistence"]),

                    round(core.iloc[-1]["MACD_HIST"], 3),

                    round(core.iloc[-1]["MACD_SLOPE"], 3),

                    round(core.iloc[-1]["DI_PLUS"], 1),

                    round(core.iloc[-1]["DI_MINUS"], 1),

                    round(core.iloc[-1]["DMI_ACCEL"], 2),

                    round(ret_structure, 2)

                ])

        except:

            pass

# ============================================================

# DATAFRAMES

# ============================================================

trade_bt = pd.DataFrame(

    trade_rows,

    columns=[

        "Date","Ticker","RSI","RS20","RS60","Persistence",

        "MACD_HIST","MACD_SLOPE","DI_PLUS","DI_MINUS",

        "DMI_ACCEL","Return_10D"

    ]

)

structure_bt = pd.DataFrame(

    structure_rows,

    columns=[

        "Date","Ticker","RSI","RS20","RS60","Persistence",

        "MACD_HIST","MACD_SLOPE","DI_PLUS","DI_MINUS",

        "DMI_ACCEL","Return_40D"

    ]

)

# ============================================================

# SIGNAL GAP FILTER

# ============================================================

def apply_signal_gap(signal_df, date_col="Date", ticker_col="Ticker", min_gap=5):

    if signal_df.empty:

        return signal_df

    signal_df = signal_df.sort_values([ticker_col, date_col]).copy()

    keep_rows = []

    last_signal_date = {}

    for idx, row in signal_df.iterrows():

        ticker = row[ticker_col]

        d = pd.to_datetime(row[date_col])

        if ticker not in last_signal_date:

            keep_rows.append(idx)

            last_signal_date[ticker] = d

        else:

            if (d - last_signal_date[ticker]).days >= min_gap:

                keep_rows.append(idx)

                last_signal_date[ticker] = d

    return signal_df.loc[keep_rows].copy()

trade_bt = apply_signal_gap(trade_bt, min_gap=MIN_GAP_DAYS)

structure_bt = apply_signal_gap(structure_bt, min_gap=MIN_GAP_DAYS)

# ============================================================

# BENCHMARK ALPHA

# ============================================================

bench["Future10"] = bench["Close"].shift(-TRADE_HOLD_DAYS)

bench["BenchReturn10D"] = (bench["Future10"] / bench["Close"] - 1) * 100

bench["Future40"] = bench["Close"].shift(-STRUCTURE_HOLD_DAYS)

bench["BenchReturn40D"] = (bench["Future40"] / bench["Close"] - 1) * 100

bench10 = bench["BenchReturn10D"]

bench40 = bench["BenchReturn40D"]

if not trade_bt.empty:

    trade_bt["Bench10D"] = trade_bt["Date"].map(bench10)

    trade_bt["Alpha10D"] = trade_bt["Return_10D"] - trade_bt["Bench10D"]

    trade_bt = trade_bt.dropna()

if not structure_bt.empty:

    structure_bt["Bench40D"] = structure_bt["Date"].map(bench40)

    structure_bt["Alpha40D"] = structure_bt["Return_40D"] - structure_bt["Bench40D"]

    structure_bt = structure_bt.dropna()

# ============================================================

# ELITE SIGNALS

# ============================================================

if not trade_bt.empty and not structure_bt.empty:

    elite = pd.merge(

        trade_bt,

        structure_bt,

        on=["Date", "Ticker"],

        suffixes=("_10D", "_40D")

    )

else:

    elite = pd.DataFrame()

# ============================================================

# SUMMARY FUNCTION

# ============================================================

def summarize(df, return_col, alpha_col, label):

    if df is None or len(df) == 0:

        return pd.DataFrame([{

            "Signal": label,

            "Trades": 0,

            "AvgReturn": np.nan,

            "MedianReturn": np.nan,

            "HitRate": np.nan,

            "AvgAlpha": np.nan,

            "MedianAlpha": np.nan,

            "Best": np.nan,

            "Worst": np.nan

        }])

    return pd.DataFrame([{

        "Signal": label,

        "Trades": len(df),

        "AvgReturn": round(df[return_col].mean(), 2),

        "MedianReturn": round(df[return_col].median(), 2),

        "HitRate": round((df[return_col] > 0).mean() * 100, 1),

        "AvgAlpha": round(df[alpha_col].mean(), 2),

        "MedianAlpha": round(df[alpha_col].median(), 2),

        "Best": round(df[return_col].max(), 2),

        "Worst": round(df[return_col].min(), 2)

    }])

# ============================================================

# SUMMARY TABLES

# ============================================================

trade_summary = summarize(trade_bt, "Return_10D", "Alpha10D", "10D TRADE")

structure_summary = summarize(structure_bt, "Return_40D", "Alpha40D", "40D STRUCTURE")

if not elite.empty:

    elite10_summary = summarize(elite, "Return_10D", "Alpha10D", "ELITE 10D")

    elite40_summary = summarize(elite, "Return_40D", "Alpha40D", "ELITE 40D")

else:

    elite10_summary = summarize(pd.DataFrame(), "Return_10D", "Alpha10D", "ELITE 10D")

    elite40_summary = summarize(pd.DataFrame(), "Return_40D", "Alpha40D", "ELITE 40D")

summary = pd.concat([

    trade_summary,

    structure_summary,

    elite10_summary,

    elite40_summary

]).reset_index(drop=True)

# ============================================================

# LIVE MODEL

# ============================================================

live_rows = []

for t in tickers:

    try:

        df = data.get(t)

        if df is None:

            continue

        core = add_core(df.tail(250))

        if len(core) < 5:

            continue

        row = core.iloc[-1]

        trade_sig = trade_signal(core)

        struct_sig = structure_signal(core)

        if trade_sig == "NO SIGNAL" and struct_sig == "NO SIGNAL":

            continue

        if trade_sig == "10D TRADE" and struct_sig == "40D STRUCTURE":

            combined = "ELITE"

        elif trade_sig == "10D TRADE":

            combined = "TRADE"

        elif struct_sig == "40D STRUCTURE":

            combined = "STRUCTURE"

        else:

            combined = "WATCH"

        live_rows.append([

            t,

            round(row["Close"], 2),

            round(row["RSI"], 1),

            round(row["RS20"], 1),

            round(row["RS60"], 1),

            int(row["Persistence"]),

            round(row["MACD_HIST"], 3),

            round(row["MACD_SLOPE"], 3),

            round(row["DI_PLUS"], 1),

            round(row["DI_MINUS"], 1),

            round(row["DMI_ACCEL"], 2),

            combined

        ])

    except:

        pass

live = pd.DataFrame(

    live_rows,

    columns=[

        "Ticker","Price","RSI","RS20","RS60","Persistence",

        "MACD_HIST","MACD_SLOPE","DI_PLUS","DI_MINUS",

        "DMI_ACCEL","Signal"

    ]

)

if not live.empty:

    rank = {

        "ELITE": 1,

        "TRADE": 2,

        "STRUCTURE": 3,

        "WATCH": 4

    }

    live["Rank"] = live["Signal"].map(rank)

    live = (

        live

        .sort_values(["Rank", "Persistence", "RS20"], ascending=[True, False, False])

        .drop(columns=["Rank"])

        .reset_index(drop=True)

    )

# ============================================================

# OUTPUT

# ============================================================

print("\n============================================================")

print("🔥 POWERMODE OSLO v7.1 — FULL ALPHA TEST REGIME")

print("============================================================")

display(summary)

print("\n============================================================")

print("🔥 LIVE MODEL")

print("============================================================")

display(live)

print("\n============================================================")

print("🔥 ELITE SIGNALS")

print("============================================================")

display(elite.head(30))

# ============================================================

# EXPORT

# ============================================================

filename = f"powermode_oslo_v71_dual_engine_alpha_{TODAY}.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    trade_bt.to_excel(writer, sheet_name="10D_Trades", index=False)

    structure_bt.to_excel(writer, sheet_name="40D_Structure", index=False)

    elite.to_excel(writer, sheet_name="Elite", index=False)

    summary.to_excel(writer, sheet_name="Summary", index=False)

    live.to_excel(writer, sheet_name="Live_Model", index=False)

# ============================================================

# SAVE FOR MASTER

# ============================================================

V7_LIVE = live.copy()

print("\n✅ Saved:", filename)

print("✅ V7_LIVE stored:", V7_LIVE.shape)

ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HSHIP.OL']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')


Loaded: 67

Missing / not loaded:
['HSHIP.OL']

🔥 POWERMODE OSLO v7.1 — FULL ALPHA TEST REGIME


,Signal,Trades,AvgReturn,MedianReturn,HitRate,AvgAlpha,MedianAlpha,Best,Worst
0,10D TRADE,699,1.86,1.52,59.9,0.30,-0.00,38.40,-22.82
1,40D STRUCTURE,343,8.77,7.49,72.0,0.87,0.24,62.06,-38.37
2,ELITE 10D,294,1.77,1.48,59.5,0.06,-0.37,25.41,-22.82
3,ELITE 40D,294,8.88,7.53,72.4,1.08,0.62,62.06,-38.37



🔥 LIVE MODEL


,Ticker,Price,RSI,RS20,RS60,Persistence,MACD_HIST,MACD_SLOPE,DI_PLUS,DI_MINUS,DMI_ACCEL,Signal
0,BWLPG.OL,195.40,67.6,14.7,23.4,19,0.660,0.179,26.2,11.0,0.43,ELITE
1,KIT.OL,108.60,61.7,9.3,17.7,17,0.475,0.019,32.0,14.7,-3.74,ELITE
2,HEX.OL,10.73,56.3,5.2,31.5,14,-0.160,0.106,30.8,26.2,22.56,ELITE
3,SATS.OL,43.05,54.3,-1.4,3.1,15,0.144,0.070,30.2,19.9,1.58,TRADE
4,SUBC.OL,346.00,67.7,13.1,37.4,13,0.053,1.407,38.6,14.7,10.06,TRADE
5,PEN.OL,35.55,63.0,13.0,59.7,11,0.036,0.044,33.4,20.2,1.79,TRADE
6,DOFG.OL,142.00,64.2,5.4,28.1,11,0.188,0.195,27.5,13.8,3.59,TRADE
7,AKRBP.OL,347.60,54.8,4.4,32.2,10,-1.005,0.745,31.8,27.7,5.40,TRADE
8,AFG.OL,179.40,66.5,4.4,1.1,9,0.973,0.509,39.9,14.4,3.39,TRADE
9,BNOR.OL,578.00,56.9,6.1,38.6,8,-2.035,0.097,22.0,18.6,-0.19,TRADE



🔥 ELITE SIGNALS


,Date,Ticker,RSI_10D,RS20_10D,RS60_10D,Persistence_10D,MACD_HIST_10D,MACD_SLOPE_10D,DI_PLUS_10D,DI_MINUS_10D,...,RS60_40D,Persistence_40D,MACD_HIST_40D,MACD_SLOPE_40D,DI_PLUS_40D,DI_MINUS_40D,DMI_ACCEL_40D,Return_40D,Bench40D,Alpha40D
0,2025-12-22,ABL.OL,55.7,5.3,4.0,16,0.009,0.000,20.0,13.7,...,4.0,16,0.009,0.000,20.0,13.7,1.27,13.82,12.804147,1.015853
1,2026-01-02,ABL.OL,56.6,2.6,5.2,18,0.002,0.003,17.4,10.6,...,5.2,18,0.002,0.003,17.4,10.6,0.86,11.93,11.496514,0.433486
2,2026-01-09,ABL.OL,54.0,0.7,4.5,19,-0.016,0.006,12.9,12.6,...,4.5,19,-0.016,0.006,12.9,12.6,2.41,17.21,12.197364,5.012636
3,2026-01-22,ABL.OL,57.5,0.5,4.9,16,-0.008,0.017,22.4,11.6,...,4.9,16,-0.008,0.017,22.4,11.6,18.07,27.00,15.839904,11.160096
4,2026-01-27,ABL.OL,65.9,4.4,10.3,16,0.029,0.025,37.9,11.1,...,10.3,16,0.029,0.025,37.9,11.1,16.33,23.89,12.243516,11.646484
5,2026-03-06,ABL.OL,62.0,6.8,18.0,15,-0.036,0.040,28.2,20.1,...,18.0,15,-0.036,0.040,28.2,20.1,10.80,-0.49,3.936600,-4.426600
6,2026-03-11,ABL.OL,61.6,5.6,20.0,15,0.014,0.012,38.1,14.6,...,20.0,15,0.014,0.012,38.1,14.6,13.50,0.98,3.668741,-2.688741
7,2025-10-01,AFG.OL,58.9,2.4,8.3,18,-0.377,0.188,25.8,21.3,...,8.3,18,-0.377,0.188,25.8,21.3,5.50,6.46,-2.978205,9.438205
8,2025-10-06,AFG.OL,57.2,0.5,9.4,17,-0.190,0.154,21.7,21.1,...,9.4,17,-0.190,0.154,21.7,21.1,1.96,7.05,-3.727131,10.777131
9,2025-10-21,AFG.OL,53.6,0.4,8.0,14,-0.432,0.142,28.5,26.0,...,8.0,14,-0.432,0.142,28.5,26.0,13.14,10.53,0.235270,10.294730



✅ Saved: powermode_oslo_v71_dual_engine_alpha_2026-05-16.xlsx
✅ V7_LIVE stored: (23, 12)


In [ ]:
# =========================================================

# 🔥 POWERMODE v4.2 – FULL KELTNER CHANNEL BUY / SELL ENGINE

# =========================================================

# Upgrade from v4.1:

# ✅ Real Keltner Channel: EMA20 +/- 2*ATR20

# ✅ Volatility-adjusted extension

# ✅ Same signal names as v4.1 for direct comparison

# ✅ EdgeTracker backtest

# ✅ Current BUY / SELL rankings

# =========================================================

!pip install yfinance pandas numpy openpyxl -q

import yfinance as yf

import pandas as pd

import numpy as np

from datetime import date

import warnings

warnings.filterwarnings("ignore")

# =========================================================

# SETTINGS

# =========================================================

HOLD_DAYS = 10

PERIOD = "3y"

KELTNER_ATR_MULT = 2.0

# =========================================================

# FULL OSE UNIVERSE

# =========================================================

tickers = [

    "AKRBP.OL","EQNR.OL","SUBC.OL","OET.OL","PEN.OL",

    "DNO.OL","VAR.OL","BWO.OL","SBX.OL","NORAM.OL",

    "AKSO.OL","BWE.OL","BNOR.OL","TGS.OL","DOFG.OL",

    "SOFF.OL","SDRL.OL","PNOR.OL",

    "MPCC.OL","HAUTO.OL","BWLPG.OL",

    "2020.OL","SFL.OL","FRO.OL","HAFNI.OL","HSHIP.OL","DVD.OL",

    "NHY.OL","YAR.OL","ELK.OL","TOM.OL","BORR.OL",

    "AFG.OL","VEI.OL","KCC.OL","KIT.OL",

    "DNB.OL","KOG.OL","STB.OL","GJF.OL",

    "MING.OL","PROT.OL","SBNOR.OL","BONHR.OL",

    "MOWI.OL","SALM.OL","LSG.OL","AKH.OL","AUSS.OL",

    "OLT.OL","SNTIA.OL",

    "ATEA.OL","LINK.OL","NORBT.OL",

    "NOD.OL","KID.OL","NAPA.OL","NORCO.OL","MULTI.OL",

    "DFENS.OL",

    "AUTO.OL","ODL.OL","BELCO.OL","PARB.OL",

    "ARR.OL","EPR.OL","RECSI.OL","WAWI.OL",

    "SATS.OL","CADLR.OL","CMBTO.OL",

    "ENVIP.OL","ARCH.OL","SKUE.OL","ABL.OL",

    "SMOP.OL","SPOL.OL","HSHP.OL","SOAG.OL","WWI.OL","WWIB.OL",

    "SCATC.OL","HEX.OL","MPCES.OL",

    "TEL.OL"

]

# Remove duplicates and Yahoo-broken names

tickers = sorted(list(set(tickers)))

bad_tickers = [

    "GOGL.OL",

    "CRAYN.OL",

    "XXL.OL"

]

tickers = [t for t in tickers if t not in bad_tickers]

# =========================================================

# DOWNLOAD DATA

# =========================================================

print("Downloading data...")

raw = yf.download(

    tickers,

    period=PERIOD,

    auto_adjust=True,

    progress=False,

    group_by="ticker",

    threads=False

)

data = {}

for t in tickers:

    try:

        df = raw[t].dropna()

        if isinstance(df.columns, pd.MultiIndex):

            df.columns = df.columns.get_level_values(0)

        df = df[["Open","High","Low","Close","Volume"]].dropna()

        if len(df) > 220:

            data[t] = df

    except Exception:

        pass

print("Loaded:", len(data))

missing = [t for t in tickers if t not in data]

print("\nMissing / not loaded:")

print(missing)

# =========================================================

# INDICATORS

# =========================================================

def ema(s, span):

    return s.ewm(span=span, adjust=False).mean()

def true_range(df):

    return pd.concat([

        df["High"] - df["Low"],

        (df["High"] - df["Close"].shift()).abs(),

        (df["Low"] - df["Close"].shift()).abs()

    ], axis=1).max(axis=1)

def kama(series, er_period=10, fast=2, slow=30):

    change = abs(series - series.shift(er_period))

    volatility = abs(series.diff()).rolling(er_period).sum()

    er = change / volatility

    sc = (

        er * (2/(fast+1) - 2/(slow+1))

        + 2/(slow+1)

    ) ** 2

    out = series.copy()

    out.iloc[:er_period] = series.iloc[:er_period]

    for i in range(er_period, len(series)):

        if pd.isna(sc.iloc[i]):

            out.iloc[i] = out.iloc[i-1]

        else:

            out.iloc[i] = out.iloc[i-1] + sc.iloc[i] * (series.iloc[i] - out.iloc[i-1])

    return out

def rsi(series, period=21):

    delta = series.diff()

    gain = delta.clip(lower=0).rolling(period).mean()

    loss = (-delta.clip(upper=0)).rolling(period).mean()

    rs = gain / loss

    return 100 - (100 / (1 + rs))

def dmi(df, period=10):

    high = df["High"]

    low = df["Low"]

    close = df["Close"]

    plus_dm_raw = high.diff()

    minus_dm_raw = -low.diff()

    plus_dm = np.where(

        (plus_dm_raw > minus_dm_raw) & (plus_dm_raw > 0),

        plus_dm_raw,

        0

    )

    minus_dm = np.where(

        (minus_dm_raw > plus_dm_raw) & (minus_dm_raw > 0),

        minus_dm_raw,

        0

    )

    tr = true_range(df)

    atr = tr.rolling(period).mean()

    plus_di = 100 * pd.Series(plus_dm, index=df.index).rolling(period).sum() / atr

    minus_di = 100 * pd.Series(minus_dm, index=df.index).rolling(period).sum() / atr

    return plus_di, minus_di

# =========================================================

# FEATURE ENGINE — v4.2 FULL KELTNER

# =========================================================

def build_features(df):

    df = df.copy()

    # -----------------------------------------------------

    # KELTNER CHANNEL

    # -----------------------------------------------------

    df["atr20"] = true_range(df).rolling(20).mean()

    df["keltner_mid"] = ema(df["Close"], 20)

    df["keltner_upper"] = df["keltner_mid"] + KELTNER_ATR_MULT * df["atr20"]

    df["keltner_lower"] = df["keltner_mid"] - KELTNER_ATR_MULT * df["atr20"]

    df["keltner_mid_up"] = df["keltner_mid"] > df["keltner_mid"].shift(5)

    df["price_above_keltner_mid"] = df["Close"] > df["keltner_mid"]

    df["price_above_keltner_upper"] = df["Close"] > df["keltner_upper"]

    df["price_below_keltner_lower"] = df["Close"] < df["keltner_lower"]

    df["keltner_position"] = (

        (df["Close"] - df["keltner_lower"]) /

        (df["keltner_upper"] - df["keltner_lower"])

    )

    df["keltner_extension_pct"] = (

        (df["Close"] / df["keltner_upper"] - 1) * 100

    )

    # -----------------------------------------------------

    # KAMA

    # -----------------------------------------------------

    df["kama"] = kama(df["Close"])

    df["kama_up"] = df["kama"] > df["kama"].shift(5)

    df["price_above_kama"] = df["Close"] > df["kama"]

    df["distance_kama"] = (

        (df["Close"] / df["kama"] - 1) * 100

    )

    # -----------------------------------------------------

    # MACD

    # -----------------------------------------------------

    df["macd"] = ema(df["Close"], 12) - ema(df["Close"], 26)

    df["macd_signal"] = ema(df["macd"], 9)

    df["macd_above_signal"] = df["macd"] > df["macd_signal"]

    df["macd_hist"] = df["macd"] - df["macd_signal"]

    df["macd_hist_slope"] = df["macd_hist"] - df["macd_hist"].shift(1)

    df["macd_hist_pivot_up"] = (

        (df["macd_hist"] > df["macd_hist"].shift(1)) &

        (df["macd_hist"].shift(1) < df["macd_hist"].shift(2))

    )

    # -----------------------------------------------------

    # DMI

    # -----------------------------------------------------

    df["dmi_plus"], df["dmi_minus"] = dmi(df)

    df["dmi_plus_gt_minus"] = df["dmi_plus"] > df["dmi_minus"]

    df["dmi_minus_gt_plus"] = df["dmi_minus"] > df["dmi_plus"]

    df["dmi_spread"] = df["dmi_plus"] - df["dmi_minus"]

    df["dmi_spread_acceleration"] = df["dmi_spread"] - df["dmi_spread"].shift(1)

    df["dmi_spread_accelerating"] = df["dmi_spread_acceleration"] > 0

    # -----------------------------------------------------

    # RSI

    # -----------------------------------------------------

    df["rsi21"] = rsi(df["Close"], 21)

    df["rsi21_pivot_up"] = (

        (df["rsi21"] > df["rsi21"].shift(1)) &

        (df["rsi21"].shift(1) < df["rsi21"].shift(2))

    )

    df["rsi21_pivot_down"] = (

        (df["rsi21"] < df["rsi21"].shift(1)) &

        (df["rsi21"].shift(1) > df["rsi21"].shift(2))

    )

    # -----------------------------------------------------

    # STRUCTURE

    # -----------------------------------------------------

    df["price_above_keltner"] = df["price_above_keltner_mid"]

    df["holds_structure"] = (

        df["price_above_kama"] |

        df["price_above_keltner_mid"]

    )

    df["strong_structure"] = (

        df["price_above_kama"] &

        df["price_above_keltner_mid"] &

        df["kama_up"] &

        df["keltner_mid_up"]

    )

    # -----------------------------------------------------

    # FULL KELTNER EXTENSION LOGIC

    # -----------------------------------------------------

    df["mild_extension"] = (

        df["price_above_keltner_upper"] |

        (df["keltner_position"] > 1.0)

    )

    df["extreme_extension"] = (

        df["price_above_keltner_upper"] &

        (df["distance_kama"] > 8) &

        (df["keltner_extension_pct"] > 1.5)

    )

    df["healthy_extension"] = (

        (df["keltner_position"] >= 0.55) &

        (df["keltner_position"] <= 1.05) &

        (df["distance_kama"] <= 8)

    )

    # -----------------------------------------------------

    # RESET / RELOAD

    # -----------------------------------------------------

    df["momentum_reset"] = (

        (df["macd_hist"].shift(1) < df["macd_hist"].shift(4)) |

        (df["rsi21"].shift(1) < df["rsi21"].shift(4)) |

        (df["dmi_spread"].shift(1) < df["dmi_spread"].shift(4))

    )

    # -----------------------------------------------------

    # PIVOT

    # -----------------------------------------------------

    df["pivot_trigger"] = (

        (

            df["macd_hist_pivot_up"] |

            (df["macd_hist_slope"] > 0)

        ) &

        (

            df["dmi_spread_accelerating"] |

            df["rsi21_pivot_up"]

        ) &

        df["holds_structure"]

    )

    # -----------------------------------------------------

    # RE-ACCELERATION

    # -----------------------------------------------------

    df["reacceleration"] = (

        (df["macd_hist_slope"] > 0) &

        (

            df["dmi_spread_accelerating"] |

            (df["dmi_spread"] > df["dmi_spread"].shift(2))

        )

    )

    # -----------------------------------------------------

    # BREAKDOWN / EXIT

    # -----------------------------------------------------

    df["breakdown"] = (

        (

            ~df["price_above_kama"] &

            ~df["price_above_keltner_mid"]

        )

        |

        (

            ~df["macd_above_signal"] &

            df["dmi_minus_gt_plus"]

        )

        |

        (

            df["price_below_keltner_lower"] &

            df["dmi_minus_gt_plus"]

        )

    )

    return df

# =========================================================

# PROCESS

# =========================================================

processed = {}

for ticker in data.keys():

    daily = build_features(data[ticker])

    weekly = data[ticker].resample("W-FRI").agg({

        "Open":"first",

        "High":"max",

        "Low":"min",

        "Close":"last",

        "Volume":"sum"

    }).dropna()

    weekly = build_features(weekly)

    weekly_cols = [

        "kama_up",

        "keltner_mid_up",

        "macd_above_signal",

        "dmi_plus_gt_minus",

        "strong_structure"

    ]

    weekly = weekly[weekly_cols].add_prefix("weekly_")

    weekly = weekly.reindex(

        daily.index,

        method="ffill"

    )

    daily = pd.concat([daily, weekly], axis=1)

    # -----------------------------------------------------

    # SCORES

    # -----------------------------------------------------

    daily["weekly_score"] = (

        daily[[

            "weekly_kama_up",

            "weekly_keltner_mid_up",

            "weekly_macd_above_signal",

            "weekly_dmi_plus_gt_minus"

        ]]

        .fillna(False)

        .astype(int)

        .sum(axis=1)

        / 4

    )

    daily["daily_score"] = (

        daily[[

            "kama_up",

            "keltner_mid_up",

            "macd_above_signal",

            "dmi_plus_gt_minus",

            "pivot_trigger",

            "reacceleration"

        ]]

        .fillna(False)

        .astype(int)

        .sum(axis=1)

        / 6

    )

    daily["hybrid_score"] = (

        daily["daily_score"] * 0.40 +

        daily["weekly_score"] * 0.60

    )

    # -----------------------------------------------------

    # EDGE SCORE v4.2

    # -----------------------------------------------------

    daily["edge_score"] = (

          daily["weekly_score"] * 35

        + daily["daily_score"] * 35

        + daily["pivot_trigger"].astype(int) * 15

        + daily["reacceleration"].astype(int) * 15

        + daily["healthy_extension"].astype(int) * 5

        - daily["extreme_extension"].astype(int) * 25

        - (

            daily["price_above_keltner_upper"] &

            (daily["macd_hist_slope"] < 0)

          ).astype(int) * 10

    )

    # -----------------------------------------------------

    # SIGNALS — same names as v4.1

    # -----------------------------------------------------

    def signal(row):

        # ---------------- BUY ----------------

        if (

            row["weekly_score"] == 1

            and row["pivot_trigger"]

            and row["reacceleration"]

            and not row["extreme_extension"]

            and row["healthy_extension"]

        ):

            return "FULL QUALITY BUY"

        if (

            row["weekly_score"] >= 0.75

            and row["pivot_trigger"]

            and not row["extreme_extension"]

            and row["holds_structure"]

        ):

            return "POWERMODE CORE"

        if (

            row["weekly_score"] >= 0.75

            and row["momentum_reset"]

            and row["reacceleration"]

            and not row["breakdown"]

            and not row["extreme_extension"]

        ):

            return "HEALTHY PULLBACK"

        # ---------------- SELL ----------------

        if (

            row["extreme_extension"]

            and row["macd_hist_slope"] < 0

        ):

            return "TRIM / TAKE PROFIT"

        if (

            row["rsi21_pivot_down"]

            and row["macd_hist_slope"] < 0

            and (

                row["dmi_minus_gt_plus"]

                or row["price_above_keltner_upper"]

            )

        ):

            return "EXIT WARNING"

        if row["breakdown"]:

            return "EXIT NOW"

        return "NEUTRAL"

    daily["signal"] = daily.apply(signal, axis=1)

    processed[ticker] = daily

# =========================================================

# EDGE TRACKER

# =========================================================

all_rows = []

for ticker in processed.keys():

    df = processed[ticker].copy()

    df["ticker"] = ticker

    df["future_close"] = df["Close"].shift(-HOLD_DAYS)

    df["return_10d"] = (

        (df["future_close"] / df["Close"] - 1) * 100

    )

    all_rows.append(df)

all_df = pd.concat(all_rows)

market_return = (

    all_df

    .groupby(all_df.index)["return_10d"]

    .mean()

)

all_df["market_return"] = all_df.index.map(market_return)

all_df["alpha_10d"] = (

    all_df["return_10d"] - all_df["market_return"]

)

signals = [

    "FULL QUALITY BUY",

    "POWERMODE CORE",

    "HEALTHY PULLBACK",

    "TRIM / TAKE PROFIT",

    "EXIT WARNING",

    "EXIT NOW",

    "NEUTRAL"

]

results = []

for s in signals:

    sample = all_df[

        all_df["signal"] == s

    ].dropna(subset=["return_10d"])

    if len(sample) < 20:

        continue

    results.append({

        "Signal": s,

        "Trades": len(sample),

        "AvgReturn10d": round(sample["return_10d"].mean(), 2),

        "HitRate": round((sample["return_10d"] > 0).mean() * 100, 1),

        "AvgAlpha10d": round(sample["alpha_10d"].mean(), 2),

        "MedianReturn10d": round(sample["return_10d"].median(), 2),

        "Best": round(sample["return_10d"].max(), 2),

        "Worst": round(sample["return_10d"].min(), 2)

    })

edge = pd.DataFrame(results)

edge = edge.sort_values(

    by="AvgAlpha10d",

    ascending=False

).reset_index(drop=True)

print("")

print("=================================================")

print("🔥 POWERMODE v4.2 FULL KELTNER — EDGE TRACKER")

print("=================================================")

display(edge)

# =========================================================

# LIVE BUY / SELL RANKINGS

# =========================================================

rows = []

for ticker in processed.keys():

    last = processed[ticker].dropna().iloc[-1]

    score = last["edge_score"]

    if score >= 85:

        grade = "A+"

    elif score >= 75:

        grade = "A"

    elif score >= 65:

        grade = "B"

    elif score >= 55:

        grade = "C"

    else:

        grade = "D"

    rows.append({

        "Ticker": ticker,

        "Price": round(last["Close"], 2),

        "Weekly": round(last["weekly_score"], 2),

        "Daily": round(last["daily_score"], 2),

        "Hybrid": round(last["hybrid_score"], 2),

        "EdgeScore": round(score, 1),

        "Grade": grade,

        "Pivot": bool(last["pivot_trigger"]),

        "ReAccel": bool(last["reacceleration"]),

        "Reset": bool(last["momentum_reset"]),

        "HealthyExt": bool(last["healthy_extension"]),

        "MildExt": bool(last["mild_extension"]),

        "Extended": bool(last["extreme_extension"]),

        "KeltnerPos": round(last["keltner_position"], 2),

        "KeltnerExt%": round(last["keltner_extension_pct"], 2),

        "DistKAMA%": round(last["distance_kama"], 2),

        "RSI21": round(last["rsi21"], 1),

        "Signal": last["signal"]

    })

live = pd.DataFrame(rows)

buy_list = live[

    live["Signal"].isin([

        "FULL QUALITY BUY",

        "POWERMODE CORE",

        "HEALTHY PULLBACK"

    ])

].sort_values(

    by="EdgeScore",

    ascending=False

).reset_index(drop=True)

sell_list = live[

    live["Signal"].isin([

        "TRIM / TAKE PROFIT",

        "EXIT WARNING",

        "EXIT NOW"

    ])

].sort_values(

    by="EdgeScore",

    ascending=True

).reset_index(drop=True)

print("")

print("=================================================")

print("🔥 CURRENT TOP BUYS — v4.2 FULL KELTNER")

print("=================================================")

display(buy_list.head(25))

print("")

print("=================================================")

print("🔥 CURRENT SELL / EXIT LIST — v4.2 FULL KELTNER")

print("=================================================")

display(sell_list.head(25))

# =========================================================

# EXPORT

# =========================================================

filename = f"powermode_v42_full_keltner_{date.today()}.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:

    edge.to_excel(writer, sheet_name="EdgeTracker", index=False)

    buy_list.to_excel(writer, sheet_name="TopBuys", index=False)

    sell_list.to_excel(writer, sheet_name="SellExit", index=False)

    live.to_excel(writer, sheet_name="LiveAll", index=False)

print("")

print("✅ Saved:", filename)

# =========================================================
# STORE FOR MASTER
# =========================================================

V42_BUY = buy_list.copy()
V42_SELL = sell_list.copy()
V42_LIVE = live.copy()
V42_EDGE = edge.copy()

print("")
print("✅ V42 stored for MASTER")
print("BUY:", V42_BUY.shape)
print("SELL:", V42_SELL.shape)
print("LIVE:", V42_LIVE.shape)
print("EDGE:", V42_EDGE.shape)


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: BELCO.OL"}}}
ERROR:yfinance:
6 Failed downloads:
ERROR:yfinance:['BELCO.OL', 'SDRL.OL', 'OLT.OL', 'HSHIP.OL', 'SBX.OL', 'SFL.OL']: YFPricesMissingError('possibly delisted; no price data found  (period=3y) (Yahoo error = "No data found, symbol may be delisted")')


Loaded: 77

Missing / not loaded:
['BELCO.OL', 'BORR.OL', 'CMBTO.OL', 'HSHIP.OL', 'OLT.OL', 'SBX.OL', 'SDRL.OL', 'SFL.OL']

🔥 POWERMODE v4.2 FULL KELTNER — EDGE TRACKER


,Signal,Trades,AvgReturn10d,HitRate,AvgAlpha10d,MedianReturn10d,Best,Worst
0,POWERMODE CORE,4517,0.96,54.4,0.13,0.62,54.71,-34.76
1,EXIT WARNING,3918,1.12,55.6,0.04,0.82,69.00,-97.39
2,NEUTRAL,21306,0.72,53.4,-0.01,0.50,129.24,-96.02
3,EXIT NOW,21757,0.99,56.4,-0.01,0.87,65.85,-97.48
4,FULL QUALITY BUY,4022,0.81,54.3,-0.04,0.64,78.40,-75.00
5,HEALTHY PULLBACK,344,0.60,53.2,-0.24,0.52,23.37,-34.75



🔥 CURRENT TOP BUYS — v4.2 FULL KELTNER


,Ticker,Price,Weekly,Daily,Hybrid,EdgeScore,Grade,Pivot,ReAccel,Reset,HealthyExt,MildExt,Extended,KeltnerPos,KeltnerExt%,DistKAMA%,RSI21,Signal
0,NORAM.OL,52.50,1.00,1.00,1.00,105.0,A+,True,True,False,True,False,False,0.99,-0.17,5.70,64.0,FULL QUALITY BUY
1,YAR.OL,533.40,1.00,1.00,1.00,105.0,A+,True,True,True,True,False,False,0.74,-3.12,2.39,53.9,FULL QUALITY BUY
2,KIT.OL,108.60,1.00,1.00,1.00,105.0,A+,True,True,True,True,False,False,0.82,-2.48,4.26,63.1,FULL QUALITY BUY
3,DOFG.OL,142.00,1.00,1.00,1.00,105.0,A+,True,True,True,True,False,False,0.86,-1.28,3.13,56.8,FULL QUALITY BUY
4,SMOP.OL,58.40,1.00,1.00,1.00,100.0,A+,True,True,False,False,False,False,0.98,-0.39,12.58,61.5,POWERMODE CORE
5,SUBC.OL,346.00,1.00,1.00,1.00,100.0,A+,True,True,True,False,True,False,1.10,1.43,7.77,60.9,POWERMODE CORE
6,AKSO.OL,44.88,1.00,1.00,1.00,100.0,A+,True,True,True,False,True,False,1.05,0.63,6.55,69.6,POWERMODE CORE
7,NOD.OL,204.00,1.00,0.83,0.93,99.2,A+,True,True,True,True,True,False,1.02,0.31,5.25,75.9,FULL QUALITY BUY
8,SNTIA.OL,75.30,1.00,0.83,0.93,99.2,A+,True,True,True,True,False,False,0.83,-1.57,2.63,55.4,FULL QUALITY BUY
9,VAR.OL,47.33,0.75,0.83,0.78,90.4,A+,True,True,True,True,False,False,0.76,-3.26,1.32,57.3,POWERMODE CORE



🔥 CURRENT SELL / EXIT LIST — v4.2 FULL KELTNER


,Ticker,Price,Weekly,Daily,Hybrid,EdgeScore,Grade,Pivot,ReAccel,Reset,HealthyExt,MildExt,Extended,KeltnerPos,KeltnerExt%,DistKAMA%,RSI21,Signal
0,GJF.OL,251.40,0.00,0.00,0.00,0.0,D,False,False,True,False,False,False,0.35,-4.68,-2.11,39.5,EXIT NOW
1,NORCO.OL,34.90,0.00,0.00,0.00,0.0,D,False,False,True,False,False,False,-0.08,-10.56,-4.85,30.6,EXIT WARNING
2,PROT.OL,453.00,0.00,0.00,0.00,0.0,D,False,False,True,False,False,False,0.16,-7.19,-1.85,26.4,EXIT WARNING
3,TOM.OL,94.30,0.00,0.17,0.07,5.8,D,False,False,True,False,False,False,0.16,-13.10,-2.85,27.7,EXIT NOW
4,ODL.OL,92.40,0.25,0.00,0.15,8.8,D,False,False,True,False,False,False,-0.02,-10.72,-5.45,20.5,EXIT NOW
5,KOG.OL,292.20,0.00,0.33,0.13,11.7,D,False,False,True,False,False,False,-0.00,-14.76,-6.62,25.1,EXIT NOW
6,SKUE.OL,331.00,0.50,0.00,0.30,17.5,D,False,False,True,False,False,False,0.35,-4.27,-0.69,45.4,EXIT NOW
7,MOWI.OL,197.40,0.00,0.17,0.07,20.8,D,False,True,True,False,False,False,0.25,-6.07,-2.14,34.7,EXIT NOW
8,SCATC.OL,104.20,0.00,0.17,0.07,20.8,D,False,True,False,False,False,False,-0.07,-12.70,-5.00,29.9,EXIT NOW
9,ENVIP.OL,49.40,0.75,0.00,0.45,26.2,D,False,False,True,False,False,False,0.23,-11.00,-3.50,34.0,EXIT NOW



✅ Saved: powermode_v42_full_keltner_2026-05-16.xlsx

✅ V42 stored for MASTER
BUY: (20, 18)
SELL: (30, 18)
LIVE: (77, 18)
EDGE: (6, 8)


In [ ]:
# ============================================================
# POWERMODE SVERIGE V8.1 – EDGE TRACKER / KORRIGERT
# One-cell version
# Scan + Historical Edge + 10D/40D backtest
# Stores: SE_SCAN, SE_BT for MASTER
# ============================================================

!pip -q install yfinance pandas_ta openpyxl

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
from datetime import datetime

# -----------------------------
# TICKERS
# -----------------------------

TICKERS = [
    "VOLV-B.ST", "ATCO-A.ST", "ATCO-B.ST", "ABB.ST", "SAND.ST",
    "EPI-A.ST", "ALFA.ST", "SKF-B.ST", "ASSA-B.ST", "INDT.ST",
    "LATO-B.ST", "LIFCO-B.ST", "ADDT-B.ST", "BEIJ-B.ST", "OEM-B.ST",
    "SEB-A.ST", "SWED-A.ST", "SHB-A.ST", "INVE-B.ST", "KINV-B.ST",
    "EQT.ST", "AZA.ST",
    "GETI-B.ST", "ARJO-B.ST", "ALIV-SDB.ST", "BIOA-B.ST", "VITR.ST",
    "EVO.ST", "SINCH.ST", "TRUE-B.ST", "MIPS.ST",
    "VIT-B.ST", "MSAB-B.ST",
    "BALD-B.ST", "CAST.ST", "FABG.ST", "WIHL.ST", "HUFV-A.ST",
    "NP3.ST", "SBB-B.ST",
    "BOL.ST", "SSAB-A.ST", "SSAB-B.ST", "HOLM-B.ST", "SCA-B.ST",
    "NIBE-B.ST", "TREL-B.ST",
    "THULE.ST", "AAK.ST", "AXFO.ST", "DOM.ST", "BHG.ST",
    "BUFAB.ST", "INSTAL.ST", "CTT.ST", "WALL-B.ST"
]

PERIOD = "2y"

# -----------------------------
# HELPERS
# -----------------------------

def flatten(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df

def clean_ohlcv(df):
    df = flatten(df).copy()
    needed = ["Open", "High", "Low", "Close", "Volume"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        return pd.DataFrame()
    df = df[needed].replace([np.inf, -np.inf], np.nan).dropna()
    return df

def add_indicators(df):
    df = clean_ohlcv(df)

    if df.empty or len(df) < 220:
        return pd.DataFrame()

    df["EMA10"] = ta.ema(df["Close"], length=10)
    df["EMA20"] = ta.ema(df["Close"], length=20)
    df["EMA50"] = ta.ema(df["Close"], length=50)
    df["EMA200"] = ta.ema(df["Close"], length=200)
    df["KAMA"] = ta.kama(df["Close"], length=10)

    macd = ta.macd(df["Close"])
    if macd is None or macd.empty or macd.shape[1] < 3:
        return pd.DataFrame()

    df["MACD"] = macd.iloc[:, 0]
    df["MACD_Hist"] = macd.iloc[:, 1]
    df["MACD_Signal"] = macd.iloc[:, 2]
    df["MACD_Accel"] = df["MACD_Hist"] - df["MACD_Hist"].shift(1)
    df["MACD_Accel_3D"] = df["MACD_Hist"] - df["MACD_Hist"].shift(3)

    df["RSI"] = ta.rsi(df["Close"], length=14)

    adx = ta.adx(df["High"], df["Low"], df["Close"], length=14)
    if adx is None or adx.empty:
        return pd.DataFrame()

    if "ADX_14" not in adx.columns or "DMP_14" not in adx.columns or "DMN_14" not in adx.columns:
        return pd.DataFrame()

    df["ADX"] = adx["ADX_14"]
    df["+DI"] = adx["DMP_14"]
    df["-DI"] = adx["DMN_14"]
    df["DI_Spread"] = df["+DI"] - df["-DI"]
    df["DMI_Accel"] = df["DI_Spread"] - df["DI_Spread"].shift(1)

    df["ATR"] = ta.atr(df["High"], df["Low"], df["Close"], length=14)
    df["ATR_%"] = df["ATR"] / df["Close"] * 100

    df["Vol_MA20"] = df["Volume"].rolling(20).mean()
    df["Volume_Ratio"] = df["Volume"] / df["Vol_MA20"]

    df["KAMA_Dist_%"] = (df["Close"] - df["KAMA"]) / df["KAMA"] * 100
    df["EMA20_Dist_%"] = (df["Close"] - df["EMA20"]) / df["EMA20"] * 100

    return df.replace([np.inf, -np.inf], np.nan).dropna()

def add_weekly(df):
    if df.empty:
        return pd.DataFrame()

    w = df.resample("W").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    if w.empty or len(w) < 40:
        return pd.DataFrame()

    w["W_EMA10"] = ta.ema(w["Close"], length=10)
    w["W_EMA30"] = ta.ema(w["Close"], length=30)
    w["W_RSI"] = ta.rsi(w["Close"], length=14)

    wm = ta.macd(w["Close"])
    if wm is None or wm.empty or wm.shape[1] < 2:
        return pd.DataFrame()

    w["W_MACD_Hist"] = wm.iloc[:, 1]
    w["W_MACD_Accel"] = w["W_MACD_Hist"] - w["W_MACD_Hist"].shift(1)

    return w.replace([np.inf, -np.inf], np.nan).dropna()

def edge_score(last, wlast):
    score = 0

    # Trend permission
    if last["Close"] > last["EMA20"]:
        score += 10
    if last["EMA20"] > last["EMA50"]:
        score += 10
    if last["Close"] > last["EMA50"]:
        score += 10
    if last["EMA50"] > last["EMA200"]:
        score += 6

    # Main edge: acceleration
    if last["MACD_Hist"] > 0:
        score += 10
    if last["MACD_Accel"] > 0:
        score += 20
    if last["MACD_Accel_3D"] > 0:
        score += 16

    # RSI
    if 52 <= last["RSI"] <= 68:
        score += 20
    elif 45 <= last["RSI"] < 52:
        score += 6
    elif 68 < last["RSI"] <= 72 and last["MACD_Accel"] > 0:
        score += 4
    elif last["RSI"] > 72:
        score -= 15

    # DMI
    if last["+DI"] > last["-DI"]:
        score += 12
    if last["DMI_Accel"] > 0:
        score += 12
    if last["ADX"] > 20 and last["+DI"] > last["-DI"]:
        score += 6

    # Asymmetry / extension
    if 0 <= last["KAMA_Dist_%"] <= 8:
        score += 14
    elif 8 < last["KAMA_Dist_%"] <= 12:
        score += 2
    elif last["KAMA_Dist_%"] > 12:
        score -= 20
    elif last["KAMA_Dist_%"] < -3:
        score -= 10

    # Volume
    if last["Volume_Ratio"] > 1.1:
        score += 5
    if last["Volume_Ratio"] > 1.6:
        score += 5

    # Weekly overlay
    if wlast["Close"] > wlast["W_EMA10"]:
        score += 4
    if wlast["W_EMA10"] > wlast["W_EMA30"]:
        score += 6
    if wlast["W_MACD_Hist"] > 0:
        score += 4
    if wlast["W_MACD_Accel"] > 0:
        score += 6

    # Exhaustion penalties
    if last["RSI"] > 68 and last["MACD_Accel"] < 0:
        score -= 25
    if last["RSI"] > 70 and last["DMI_Accel"] < 0:
        score -= 18
    if last["KAMA_Dist_%"] > 10 and last["MACD_Accel"] < 0:
        score -= 20
    if wlast["W_MACD_Accel"] < 0 and last["MACD_Accel"] < 0:
        score -= 15
    if last["MACD_Accel_3D"] < 0 and last["DMI_Accel"] < 0:
        score -= 15

    return round(score, 1)

def strict_signal(row):
    # Hard downgrade rules
    if row["RSI"] > 68 and row["MACD_Accel"] < 0:
        return "EXHAUSTION WATCH"

    if row["KAMA_Dist_%"] > 12 and row["MACD_Accel"] < 0:
        return "EXHAUSTION WATCH"

    if row["MACD_Accel"] < 0 and row["MACD_Accel_3D"] < 0:
        return "WATCH"

    # True EDGE must have acceleration now + 3D
    if (
        row["EdgeScore"] >= 90
        and row["MACD_Accel"] > 0
        and row["MACD_Accel_3D"] > 0
        and row["DMI_Accel"] > 0
        and row["+DI"] > row["-DI"]
        and 52 <= row["RSI"] <= 68
        and 0 <= row["KAMA_Dist_%"] <= 10
    ):
        return "EDGE LEADER"

    if (
        row["EdgeScore"] >= 75
        and row["MACD_Accel"] > 0
        and row["+DI"] > row["-DI"]
        and row["RSI"] <= 68
        and row["KAMA_Dist_%"] <= 10
    ):
        return "POWER CORE"

    if (
        row["MACD_Accel"] > 0
        and row["DMI_Accel"] > 0
        and 45 <= row["RSI"] <= 60
        and row["Close"] > row["EMA50"]
    ):
        return "EARLY EDGE"

    if row["EdgeScore"] < 45:
        return "AVOID"

    return "WATCH"

def management(signal):
    if signal == "EDGE LEADER":
        return "BUY / HOLD"
    if signal == "POWER CORE":
        return "BUY SMALL / HOLD"
    if signal == "EARLY EDGE":
        return "WATCH / STARTER"
    if signal == "EXHAUSTION WATCH":
        return "DO NOT CHASE"
    if signal == "AVOID":
        return "AVOID"
    return "WAIT"

def comment(signal):
    comments = {
        "EDGE LEADER": "Ren acceleration-edge",
        "POWER CORE": "Continuation med kjøperstyrke",
        "EARLY EDGE": "Tidlig forbedring, vent på bekreftelse",
        "EXHAUSTION WATCH": "Momentum svekkes, ikke jag",
        "AVOID": "Svak eller negativ edge",
        "WATCH": "Ikke nok edge ennå"
    }
    return comments.get(signal, "")

def v81_signal_series(df):
    return (
        (df["Close"] > df["EMA20"])
        & (df["EMA20"] > df["EMA50"])
        & (df["Close"] > df["EMA50"])
        & (df["MACD_Accel"] > 0)
        & (df["MACD_Accel_3D"] > 0)
        & (df["RSI"].between(52, 68))
        & (df["+DI"] > df["-DI"])
        & (df["DMI_Accel"] > 0)
        & (df["KAMA_Dist_%"].between(0, 10))
        & ~((df["RSI"] > 68) & (df["MACD_Accel"] < 0))
        & ~((df["KAMA_Dist_%"] > 10) & (df["MACD_Accel"] < 0))
    )

# -----------------------------
# DOWNLOAD + COMPUTE
# -----------------------------

scan_rows = []
bt_rows = []
failed = []

for ticker in TICKERS:
    try:
        raw = yf.download(
            ticker,
            period=PERIOD,
            interval="1d",
            auto_adjust=True,
            progress=False,
            threads=False
        )

        raw = clean_ohlcv(raw)

        if raw.empty or len(raw) < 260:
            failed.append((ticker, "empty/insufficient raw data"))
            continue

        df = add_indicators(raw)
        w = add_weekly(df)

        if df.empty or w.empty:
            failed.append((ticker, "indicator/weekly empty"))
            continue

        last = df.iloc[-1]
        wlast = w.iloc[-1]

        score = edge_score(last, wlast)

        row = {
            "Ticker": ticker,
            "Close": round(last["Close"], 2),
            "EdgeScore": score,
            "RSI": round(last["RSI"], 1),
            "KAMA_Dist_%": round(last["KAMA_Dist_%"], 1),
            "MACD_Accel": round(last["MACD_Accel"], 3),
            "MACD_Accel_3D": round(last["MACD_Accel_3D"], 3),
            "+DI": round(last["+DI"], 1),
            "-DI": round(last["-DI"], 1),
            "DMI_Accel": round(last["DMI_Accel"], 2),
            "ADX": round(last["ADX"], 1),
            "Volume_Ratio": round(last["Volume_Ratio"], 2),
            "W_MACD_Accel": round(wlast["W_MACD_Accel"], 3),
            "EMA20": round(last["EMA20"], 2),
            "EMA50": round(last["EMA50"], 2),
            "EMA200": round(last["EMA200"], 2),
        }

        row["Signal"] = strict_signal(row)
        row["Management"] = management(row["Signal"])
        row["EDGE TRACKER"] = comment(row["Signal"])

        # Backtest
        df["Signal"] = v81_signal_series(df)
        df["Return_10D"] = (df["Close"].shift(-10) / df["Close"] - 1) * 100
        df["Return_40D"] = (df["Close"].shift(-40) / df["Close"] - 1) * 100

        sig = df[df["Signal"]].copy()

        if len(sig) > 0:
            avg10 = sig["Return_10D"].mean()
            avg40 = sig["Return_40D"].mean()
            hr10 = (sig["Return_10D"] > 0).mean() * 100
            hr40 = (sig["Return_40D"] > 0).mean() * 100

            hist_score = (
                max(min(avg40, 30), -20) * 1.5
                + (hr40 - 50) * 0.8
                + max(min(avg10, 12), -8) * 1.0
            )

            row["Signals"] = len(sig)
            row["Avg_10D"] = round(avg10, 2)
            row["HitRate_10D"] = round(hr10, 1)
            row["Avg_40D"] = round(avg40, 2)
            row["HitRate_40D"] = round(hr40, 1)
            row["HistoricalEdge"] = round(hist_score, 1)

            bt_rows.append({
                "Ticker": ticker,
                "Signals": len(sig),
                "Avg_10D": round(avg10, 2),
                "HitRate_10D": round(hr10, 1),
                "Avg_40D": round(avg40, 2),
                "HitRate_40D": round(hr40, 1),
                "HistoricalEdge": round(hist_score, 1)
            })

        else:
            row["Signals"] = 0
            row["Avg_10D"] = np.nan
            row["HitRate_10D"] = np.nan
            row["Avg_40D"] = np.nan
            row["HitRate_40D"] = np.nan
            row["HistoricalEdge"] = -20

        row["CompositeRank"] = round(row["EdgeScore"] + row["HistoricalEdge"], 1)
        scan_rows.append(row)

    except Exception as e:
        failed.append((ticker, str(e)))
        print(f"Feil på {ticker}: {e}")

# -----------------------------
# OUTPUT
# -----------------------------

scan = pd.DataFrame(scan_rows)

display_cols = [
    "Ticker", "Close", "CompositeRank", "EdgeScore", "HistoricalEdge",
    "Signal", "Management", "EDGE TRACKER",
    "Signals", "Avg_10D", "HitRate_10D", "Avg_40D", "HitRate_40D",
    "RSI", "KAMA_Dist_%", "MACD_Accel", "MACD_Accel_3D",
    "+DI", "-DI", "DMI_Accel", "ADX", "Volume_Ratio", "W_MACD_Accel"
]

if not scan.empty:
    scan = scan.sort_values(
        by=["CompositeRank", "EdgeScore"],
        ascending=False
    ).reset_index(drop=True)

    print("POWERMODE SVERIGE V8.1 – EDGE TRACKER / KORRIGERT")
    print("Dato:", datetime.now().strftime("%Y-%m-%d %H:%M"))
    display(scan[display_cols])

bt = pd.DataFrame(bt_rows)

if not bt.empty:
    print("\nBACKTEST – V8.1 EDGE SIGNAL")
    display(
        bt.sort_values(
            by=["HistoricalEdge", "Avg_40D"],
            ascending=False
        ).reset_index(drop=True)
    )
else:
    print("Ingen backtest-signaler.")

if failed:
    print("\nTickers skipped / failed:")
    print(pd.DataFrame(failed, columns=["Ticker", "Reason"]).head(30))

# -----------------------------
# EXPORT + STORE FOR MASTER
# -----------------------------

today = datetime.now().strftime("%Y-%m-%d")
filename = f"POWERMODE_SVERIGE_V81_{today}.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    scan.to_excel(writer, sheet_name="Scan", index=False)
    bt.to_excel(writer, sheet_name="Backtest", index=False)
    pd.DataFrame(failed, columns=["Ticker", "Reason"]).to_excel(
        writer,
        sheet_name="Failed",
        index=False
    )

SE_SCAN = scan.copy()
SE_BT = bt.copy()

print("\n✅ Saved:", filename)
print("✅ SE_SCAN stored:", SE_SCAN.shape)
print("✅ SE_BT stored:", SE_BT.shape)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 12.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is in

,Ticker,Close,CompositeRank,EdgeScore,HistoricalEdge,Signal,Management,EDGE TRACKER,Signals,Avg_10D,...,RSI,KAMA_Dist_%,MACD_Accel,MACD_Accel_3D,+DI,-DI,DMI_Accel,ADX,Volume_Ratio,W_MACD_Accel
0,SHB-A.ST,130.20,153.3,130,23.3,EDGE LEADER,BUY / HOLD,Ren acceleration-edge,36,0.07,...,55.6,-0.0,0.078,0.083,25.4,18.8,1.64,15.7,0.94,-0.085
1,MSAB-B.ST,72.40,149.5,154,-4.5,EDGE LEADER,BUY / HOLD,Ren acceleration-edge,22,0.48,...,61.6,8.1,0.430,0.801,31.7,19.1,12.17,21.4,0.90,0.284
2,BOL.ST,515.20,135.8,74,61.8,WATCH,WAIT,Ikke nok edge ennå,24,2.94,...,48.0,1.6,-1.291,0.207,24.0,30.3,-7.16,19.4,2.17,1.663
3,EPI-A.ST,271.50,125.7,130,-4.3,WATCH,WAIT,Ikke nok edge ennå,14,-3.76,...,61.1,4.8,-0.372,-1.001,31.4,13.4,2.09,26.4,0.93,0.545
4,SWED-A.ST,323.10,113.3,74,39.3,EARLY EDGE,WATCH / STARTER,"Tidlig forbedring, vent på bekreftelse",43,0.69,...,50.9,-0.3,0.157,-0.092,16.3,23.2,0.58,15.7,1.04,-0.259
5,SSAB-B.ST,84.20,113.1,104,9.1,WATCH,WAIT,Ikke nok edge ennå,20,-1.69,...,57.1,2.9,-0.055,-0.437,28.0,19.0,4.92,25.4,1.20,-0.018
6,SSAB-A.ST,84.54,111.9,99,12.9,WATCH,WAIT,Ikke nok edge ennå,24,-1.49,...,57.0,2.8,-0.072,-0.461,28.0,18.6,5.02,25.3,0.94,-0.010
7,BIOA-B.ST,324.40,108.1,116,-7.9,POWER CORE,BUY SMALL / HOLD,Continuation med kjøperstyrke,27,-0.92,...,50.7,1.0,0.101,-0.494,22.1,21.2,3.43,11.1,0.73,0.337
8,OEM-B.ST,151.20,102.9,140,-37.1,POWER CORE,BUY SMALL / HOLD,Continuation med kjøperstyrke,27,-2.87,...,63.1,5.3,0.075,-0.187,31.2,10.4,0.97,38.0,0.48,0.322
9,INSTAL.ST,38.40,89.8,99,-9.2,WATCH,WAIT,Ikke nok edge ennå,17,2.39,...,59.6,3.0,-0.054,-0.243,28.3,15.2,0.08,27.7,0.70,-0.052



BACKTEST – V8.1 EDGE SIGNAL


,Ticker,Signals,Avg_10D,HitRate_10D,Avg_40D,HitRate_40D,HistoricalEdge
0,BOL.ST,24,2.94,79.2,21.44,83.3,61.8
1,SAND.ST,33,2.78,66.7,16.41,84.8,55.3
2,ABB.ST,36,3.14,72.2,10.98,80.6,44.0
3,SWED-A.ST,43,0.69,65.1,6.53,86.0,39.3
4,INVE-B.ST,23,2.08,65.2,7.74,73.9,32.8
5,SEB-A.ST,45,0.86,66.7,3.38,82.2,31.7
6,BUFAB.ST,21,1.59,66.7,5.83,71.4,27.5
7,SHB-A.ST,36,0.07,61.1,5.13,69.4,23.3
8,ALFA.ST,34,0.68,44.1,6.11,64.7,21.6
9,ASSA-B.ST,27,-0.70,37.0,1.37,66.7,14.7



✅ Saved: POWERMODE_SVERIGE_V81_2026-05-16.xlsx
✅ SE_SCAN stored: (56, 26)
✅ SE_BT stored: (54, 7)


In [ ]:
# ============================================================
# 🔥 POWERMODE MASTER CELL — NORGE + SVERIGE + V4.2 EXIT OVERLAY
# Robust one-cell version
#
# Requires previous cells to have created:
#   V7_LIVE   from Oslo V7.1
#   V42_SELL  from V4.2 Keltner sell/exit engine
#   V42_LIVE  from V4.2 Keltner full live table
#   SE_SCAN   from Sverige V8.1
#
# Optional:
#   results / US_RESULTS from USA scanner
# ============================================================

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================================
# HELPERS
# ============================================================

def is_df(x):
    return isinstance(x, pd.DataFrame) and not x.empty

def get_global_var(name):
    return globals().get(name, None)

def safe_col(df, col, default=np.nan):
    if isinstance(df, pd.DataFrame) and col in df.columns:
        return df[col]
    return default

def normalize_ticker(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def final_rank_value(signal):
    order = {
        "STRONG BUY": 1,
        "BUY": 2,
        "HOLD": 3,
        "WATCH": 4,
        "TRIM": 5,
        "EXIT": 6,
        "AVOID": 7
    }
    return order.get(str(signal), 99)

# ============================================================
# LOAD OBJECTS FROM PREVIOUS CELLS
# ============================================================

V7 = get_global_var("V7_LIVE")
V42_SELL_OBJ = get_global_var("V42_SELL")
V42_LIVE_OBJ = get_global_var("V42_LIVE")
SE = get_global_var("SE_SCAN")

# Optional USA, kept disabled/soft
USA = None
for possible_name in ["US_RESULTS", "USA_RESULTS", "results"]:
    obj = get_global_var(possible_name)
    if is_df(obj):
        USA = obj.copy()
        break

print("================================================")
print("DATA CHECK")
print("================================================")

print("V7_LIVE:", "DataFrame " + str(V7.shape) if is_df(V7) else "missing/empty")
print("V42_SELL:", "DataFrame " + str(V42_SELL_OBJ.shape) if is_df(V42_SELL_OBJ) else "missing/empty")
print("V42_LIVE:", "DataFrame " + str(V42_LIVE_OBJ.shape) if is_df(V42_LIVE_OBJ) else "missing/empty")
print("SE_SCAN:", "DataFrame " + str(SE.shape) if is_df(SE) else "missing/empty")
print("USA:", "DataFrame " + str(USA.shape) if is_df(USA) else "skipped / not active")

# ============================================================
# NORWAY MASTER — V7.1 + V4.2 EXIT OVERLAY
# ============================================================

norway_master = pd.DataFrame()

if is_df(V7):
    v7 = V7.copy()
    v7["Ticker"] = v7["Ticker"].apply(normalize_ticker)

    # V7 signal mapping
    def map_v7_signal(sig):
        sig = str(sig).upper().strip()
        if sig == "ELITE":
            return "STRONG BUY"
        if sig == "TRADE":
            return "BUY"
        if sig == "STRUCTURE":
            return "HOLD"
        if sig == "WATCH":
            return "WATCH"
        return "WATCH"

    v7["OriginalSignal"] = v7["Signal"].apply(map_v7_signal)

    # V4.2 exit overlay from sell_list
    exit_map = {}

    if is_df(V42_SELL_OBJ):
        v42_sell = V42_SELL_OBJ.copy()
        v42_sell["Ticker"] = v42_sell["Ticker"].apply(normalize_ticker)

        for _, r in v42_sell.iterrows():
            sig = str(r.get("Signal", "")).upper()

            if "EXIT NOW" in sig:
                exit_map[r["Ticker"]] = "EXIT"
            elif "EXIT WARNING" in sig:
                exit_map[r["Ticker"]] = "TRIM"
            elif "TRIM" in sig or "TAKE PROFIT" in sig:
                exit_map[r["Ticker"]] = "TRIM"

    v7["ExitOverlay"] = v7["Ticker"].map(exit_map).fillna("HOLD")

    def combine_norway(original, overlay):
        original = str(original)
        overlay = str(overlay)

        if overlay == "EXIT":
            return "EXIT"
        if overlay == "TRIM":
            if original in ["STRONG BUY", "BUY"]:
                return "TRIM"
            return "WATCH"
        return original

    v7["Final"] = [
        combine_norway(o, e)
        for o, e in zip(v7["OriginalSignal"], v7["ExitOverlay"])
    ]

    norway_master = pd.DataFrame({
        "Ticker": v7["Ticker"],
        "Country": "NO",
        "Model": "Oslo V7.1 + V4.2 Exit",
        "OriginalSignal": v7["OriginalSignal"],
        "ExitOverlay": v7["ExitOverlay"],
        "Final": v7["Final"],
        "Price": safe_col(v7, "Price"),
        "RSI": safe_col(v7, "RSI"),
        "RS20": safe_col(v7, "RS20"),
        "RS60": safe_col(v7, "RS60"),
        "Persistence": safe_col(v7, "Persistence"),
        "MACD_HIST": safe_col(v7, "MACD_HIST"),
        "MACD_SLOPE": safe_col(v7, "MACD_SLOPE"),
        "DI_PLUS": safe_col(v7, "DI_PLUS"),
        "DI_MINUS": safe_col(v7, "DI_MINUS"),
        "DMI_ACCEL": safe_col(v7, "DMI_ACCEL")
    })

    norway_master["FinalRank"] = norway_master["Final"].apply(final_rank_value)
    norway_master = norway_master.sort_values(
        by=["FinalRank", "Persistence", "RS20"],
        ascending=[True, False, False]
    ).drop(columns=["FinalRank"]).reset_index(drop=True)

else:
    print("WARNING: V7_LIVE missing. Norway master skipped.")

# ============================================================
# SWEDEN MASTER — V8.1 EDGE
# ============================================================

sweden_master = pd.DataFrame()

if is_df(SE):
    se = SE.copy()
    se["Ticker"] = se["Ticker"].apply(normalize_ticker)

    def map_sweden_final(sig):
        sig = str(sig).upper().strip()

        if sig == "EDGE LEADER":
            return "STRONG BUY"
        if sig == "POWER CORE":
            return "BUY"
        if sig == "EARLY EDGE":
            return "WATCH"
        if sig == "EXHAUSTION WATCH":
            return "TRIM"
        if sig == "AVOID":
            return "EXIT"
        return "WATCH"

    se["Final"] = se["Signal"].apply(map_sweden_final)

    edge_tracker_col = "EDGE TRACKER" if "EDGE TRACKER" in se.columns else None

    sweden_master = pd.DataFrame({
        "Ticker": se["Ticker"],
        "Country": "SE",
        "Model": "Sverige V8.1 Edge",
        "OriginalSignal": se["Signal"],
        "ExitOverlay": "",
        "Final": se["Final"],
        "Close": safe_col(se, "Close"),
        "CompositeRank": safe_col(se, "CompositeRank"),
        "EdgeScore": safe_col(se, "EdgeScore"),
        "HistoricalEdge": safe_col(se, "HistoricalEdge"),
        "Management": safe_col(se, "Management"),
        "EDGE_TRACKER": se[edge_tracker_col] if edge_tracker_col else "",
        "Signals": safe_col(se, "Signals"),
        "Avg_10D": safe_col(se, "Avg_10D"),
        "HitRate_10D": safe_col(se, "HitRate_10D"),
        "Avg_40D": safe_col(se, "Avg_40D"),
        "HitRate_40D": safe_col(se, "HitRate_40D"),
        "RSI": safe_col(se, "RSI"),
        "KAMA_Dist_%": safe_col(se, "KAMA_Dist_%"),
        "MACD_Accel": safe_col(se, "MACD_Accel"),
        "MACD_Accel_3D": safe_col(se, "MACD_Accel_3D"),
        "DMI_Accel": safe_col(se, "DMI_Accel"),
        "ADX": safe_col(se, "ADX"),
        "Volume_Ratio": safe_col(se, "Volume_Ratio")
    })

    sweden_master["FinalRank"] = sweden_master["Final"].apply(final_rank_value)
    sweden_master = sweden_master.sort_values(
        by=["FinalRank", "CompositeRank", "EdgeScore"],
        ascending=[True, False, False]
    ).drop(columns=["FinalRank"]).reset_index(drop=True)

else:
    print("WARNING: SE_SCAN missing. Sweden master skipped.")

# ============================================================
# OPTIONAL USA MASTER — SOFT / DISABLED IF NOT DATAFRAME
# ============================================================

usa_master = pd.DataFrame()

if is_df(USA):
    us = USA.copy()

    if "Ticker" in us.columns and "Signal" in us.columns:
        us["Ticker"] = us["Ticker"].apply(normalize_ticker)

        def map_us_final(sig):
            sig = str(sig).upper().strip()

            if sig == "FULL QUALITY":
                return "STRONG BUY"
            if sig in ["ACCELERATING LEADER", "POWERMODE CORE"]:
                return "BUY"
            if sig in ["WATCH / STRETCHED", "RECOVERY WATCH", "NEUTRAL WATCH"]:
                return "WATCH"
            if sig == "AVOID":
                return "EXIT"
            return "WATCH"

        us["Final"] = us["Signal"].apply(map_us_final)

        usa_master = pd.DataFrame({
            "Ticker": us["Ticker"],
            "Country": "US",
            "Model": "USA Fixed",
            "OriginalSignal": us["Signal"],
            "ExitOverlay": "",
            "Final": us["Final"],
            "Close": safe_col(us, "Close"),
            "HybridScore": safe_col(us, "HybridScore"),
            "DailyScore": safe_col(us, "DailyScore"),
            "WeeklyScore": safe_col(us, "WeeklyScore"),
            "EdgeScore": safe_col(us, "EdgeScore"),
            "EdgeSignal": safe_col(us, "EdgeSignal"),
            "RSI": safe_col(us, "RSI"),
            "ADX": safe_col(us, "ADX"),
            "Dist_KAMA_%": safe_col(us, "Dist_KAMA_%"),
            "Dist_EMA200_%": safe_col(us, "Dist_EMA200_%"),
            "VolumeExpansion": safe_col(us, "VolumeExpansion")
        })

        usa_master["FinalRank"] = usa_master["Final"].apply(final_rank_value)
        usa_master = usa_master.sort_values(
            by=["FinalRank", "HybridScore", "EdgeScore"],
            ascending=[True, False, False]
        ).drop(columns=["FinalRank"]).reset_index(drop=True)
    else:
        print("WARNING: USA object found but lacks Ticker/Signal. USA skipped.")

# ============================================================
# GLOBAL MASTER
# ============================================================

global_parts = []

if is_df(sweden_master):
    global_parts.append(sweden_master)

if is_df(usa_master):
    global_parts.append(usa_master)

if global_parts:
    global_master = pd.concat(global_parts, ignore_index=True, sort=False)
    global_master["FinalRank"] = global_master["Final"].apply(final_rank_value)

    sort_cols = ["FinalRank"]
    ascending = [True]

    if "CompositeRank" in global_master.columns:
        sort_cols.append("CompositeRank")
        ascending.append(False)
    elif "HybridScore" in global_master.columns:
        sort_cols.append("HybridScore")
        ascending.append(False)

    if "EdgeScore" in global_master.columns:
        sort_cols.append("EdgeScore")
        ascending.append(False)

    global_master = global_master.sort_values(
        by=sort_cols,
        ascending=ascending
    ).drop(columns=["FinalRank"]).reset_index(drop=True)
else:
    global_master = pd.DataFrame()

# ============================================================
# PRIORITY VIEWS
# ============================================================

norway_action = pd.DataFrame()
global_action = pd.DataFrame()
exit_list = pd.DataFrame()

if is_df(norway_master):
    norway_action = norway_master[
        norway_master["Final"].isin(["STRONG BUY", "BUY", "TRIM", "EXIT"])
    ].reset_index(drop=True)

if is_df(global_master):
    global_action = global_master[
        global_master["Final"].isin(["STRONG BUY", "BUY", "TRIM", "EXIT"])
    ].reset_index(drop=True)

exit_parts = []

if is_df(norway_master):
    exit_parts.append(
        norway_master[norway_master["Final"].isin(["TRIM", "EXIT"])].copy()
    )

if is_df(global_master):
    exit_parts.append(
        global_master[global_master["Final"].isin(["TRIM", "EXIT"])].copy()
    )

if exit_parts:
    exit_list = pd.concat(exit_parts, ignore_index=True, sort=False)
    exit_list["FinalRank"] = exit_list["Final"].apply(final_rank_value)
    exit_list = exit_list.sort_values(
        by=["FinalRank", "Country", "Ticker"],
        ascending=[True, True, True]
    ).drop(columns=["FinalRank"]).reset_index(drop=True)

# ============================================================
# DISPLAY
# ============================================================

print("\n================================================")
print("NORWAY MASTER — V7.1 + V4.2 EXIT OVERLAY")
print("================================================")

if is_df(norway_master):
    display(norway_master)
else:
    print("No Norway master data.")

print("\n================================================")
print("NORWAY ACTION LIST")
print("================================================")

if is_df(norway_action):
    display(norway_action)
else:
    print("No Norway action signals.")

print("\n================================================")
print("GLOBAL MASTER — SWEDEN V8.1 + OPTIONAL USA")
print("================================================")

if is_df(global_master):
    display(global_master)
else:
    print("No global master data.")

print("\n================================================")
print("GLOBAL ACTION LIST")
print("================================================")

if is_df(global_action):
    display(global_action)
else:
    print("No global action signals.")

print("\n================================================")
print("EXIT / TRIM LIST — ALL MARKETS")
print("================================================")

if is_df(exit_list):
    display(exit_list)
else:
    print("No exit/trim signals.")

# ============================================================
# EXPORT
# ============================================================

today = datetime.now().strftime("%Y-%m-%d")
filename = f"POWERMODE_MASTER_{today}.xlsx"

with pd.ExcelWriter(filename, engine="openpyxl") as writer:
    if is_df(norway_master):
        norway_master.to_excel(writer, sheet_name="Norway_Master", index=False)
    if is_df(norway_action):
        norway_action.to_excel(writer, sheet_name="Norway_Action", index=False)
    if is_df(sweden_master):
        sweden_master.to_excel(writer, sheet_name="Sweden_Master", index=False)
    if is_df(usa_master):
        usa_master.to_excel(writer, sheet_name="USA_Master", index=False)
    if is_df(global_master):
        global_master.to_excel(writer, sheet_name="Global_Master", index=False)
    if is_df(global_action):
        global_action.to_excel(writer, sheet_name="Global_Action", index=False)
    if is_df(exit_list):
        exit_list.to_excel(writer, sheet_name="Exit_Trim", index=False)

    # Summary sheets
    summary_rows = []

    if is_df(norway_master):
        for k, v in norway_master["Final"].value_counts().items():
            summary_rows.append({"Market": "Norway", "Final": k, "Count": int(v)})

    if is_df(global_master):
        for k, v in global_master["Final"].value_counts().items():
            summary_rows.append({"Market": "Global", "Final": k, "Count": int(v)})

    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Summary", index=False)

print("\n✅ Saved:", filename)

print("\nNORWAY SUMMARY")
if is_df(norway_master):
    print(norway_master["Final"].value_counts())

print("\nGLOBAL SUMMARY")
if is_df(global_master):
    print(global_master["Final"].value_counts())

# ============================================================
# STORE FOR LATER CELLS
# ============================================================

NORWAY_MASTER = norway_master.copy()
SWEDEN_MASTER = sweden_master.copy()
USA_MASTER = usa_master.copy()
GLOBAL_MASTER = global_master.copy()
EXIT_TRIM_LIST = exit_list.copy()

print("\n✅ Stored:")
print("NORWAY_MASTER:", NORWAY_MASTER.shape)
print("SWEDEN_MASTER:", SWEDEN_MASTER.shape)
print("USA_MASTER:", USA_MASTER.shape)
print("GLOBAL_MASTER:", GLOBAL_MASTER.shape)
print("EXIT_TRIM_LIST:", EXIT_TRIM_LIST.shape)


DATA CHECK
V7_LIVE: DataFrame (23, 12)
V42_SELL: DataFrame (30, 18)
V42_LIVE: DataFrame (77, 18)
SE_SCAN: DataFrame (56, 26)
USA: skipped / not active

NORWAY MASTER — V7.1 + V4.2 EXIT OVERLAY


,Ticker,Country,Model,OriginalSignal,ExitOverlay,Final,Price,RSI,RS20,RS60,Persistence,MACD_HIST,MACD_SLOPE,DI_PLUS,DI_MINUS,DMI_ACCEL
0,BWLPG.OL,NO,Oslo V7.1 + V4.2 Exit,STRONG BUY,HOLD,STRONG BUY,195.40,67.6,14.7,23.4,19,0.660,0.179,26.2,11.0,0.43
1,KIT.OL,NO,Oslo V7.1 + V4.2 Exit,STRONG BUY,HOLD,STRONG BUY,108.60,61.7,9.3,17.7,17,0.475,0.019,32.0,14.7,-3.74
2,HEX.OL,NO,Oslo V7.1 + V4.2 Exit,STRONG BUY,HOLD,STRONG BUY,10.73,56.3,5.2,31.5,14,-0.160,0.106,30.8,26.2,22.56
3,SATS.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,43.05,54.3,-1.4,3.1,15,0.144,0.070,30.2,19.9,1.58
4,SUBC.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,346.00,67.7,13.1,37.4,13,0.053,1.407,38.6,14.7,10.06
5,PEN.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,35.55,63.0,13.0,59.7,11,0.036,0.044,33.4,20.2,1.79
6,DOFG.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,142.00,64.2,5.4,28.1,11,0.188,0.195,27.5,13.8,3.59
7,AKRBP.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,347.60,54.8,4.4,32.2,10,-1.005,0.745,31.8,27.7,5.40
8,AFG.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,179.40,66.5,4.4,1.1,9,0.973,0.509,39.9,14.4,3.39
9,BNOR.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,578.00,56.9,6.1,38.6,8,-2.035,0.097,22.0,18.6,-0.19



NORWAY ACTION LIST


,Ticker,Country,Model,OriginalSignal,ExitOverlay,Final,Price,RSI,RS20,RS60,Persistence,MACD_HIST,MACD_SLOPE,DI_PLUS,DI_MINUS,DMI_ACCEL
0,BWLPG.OL,NO,Oslo V7.1 + V4.2 Exit,STRONG BUY,HOLD,STRONG BUY,195.40,67.6,14.7,23.4,19,0.660,0.179,26.2,11.0,0.43
1,KIT.OL,NO,Oslo V7.1 + V4.2 Exit,STRONG BUY,HOLD,STRONG BUY,108.60,61.7,9.3,17.7,17,0.475,0.019,32.0,14.7,-3.74
2,HEX.OL,NO,Oslo V7.1 + V4.2 Exit,STRONG BUY,HOLD,STRONG BUY,10.73,56.3,5.2,31.5,14,-0.160,0.106,30.8,26.2,22.56
3,SATS.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,43.05,54.3,-1.4,3.1,15,0.144,0.070,30.2,19.9,1.58
4,SUBC.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,346.00,67.7,13.1,37.4,13,0.053,1.407,38.6,14.7,10.06
5,PEN.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,35.55,63.0,13.0,59.7,11,0.036,0.044,33.4,20.2,1.79
6,DOFG.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,142.00,64.2,5.4,28.1,11,0.188,0.195,27.5,13.8,3.59
7,AKRBP.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,347.60,54.8,4.4,32.2,10,-1.005,0.745,31.8,27.7,5.40
8,AFG.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,179.40,66.5,4.4,1.1,9,0.973,0.509,39.9,14.4,3.39
9,BNOR.OL,NO,Oslo V7.1 + V4.2 Exit,BUY,HOLD,BUY,578.00,56.9,6.1,38.6,8,-2.035,0.097,22.0,18.6,-0.19



GLOBAL MASTER — SWEDEN V8.1 + OPTIONAL USA


...


GLOBAL ACTION LIST


,Ticker,Country,Model,OriginalSignal,ExitOverlay,Final,Close,CompositeRank,EdgeScore,HistoricalEdge,...,HitRate_10D,Avg_40D,HitRate_40D,RSI,KAMA_Dist_%,MACD_Accel,MACD_Accel_3D,DMI_Accel,ADX,Volume_Ratio
0,SHB-A.ST,SE,Sverige V8.1 Edge,EDGE LEADER,,STRONG BUY,130.20,153.3,130,23.3,...,61.1,5.13,69.4,55.6,-0.0,0.078,0.083,1.64,15.7,0.94
1,MSAB-B.ST,SE,Sverige V8.1 Edge,EDGE LEADER,,STRONG BUY,72.40,149.5,154,-4.5,...,36.4,1.52,40.9,61.6,8.1,0.430,0.801,12.17,21.4,0.90
2,EVO.ST,SE,Sverige V8.1 Edge,EDGE LEADER,,STRONG BUY,650.20,66.0,137,-71.0,...,37.5,-18.23,0.0,60.2,4.3,1.027,1.912,2.97,14.7,1.47
3,BIOA-B.ST,SE,Sverige V8.1 Edge,POWER CORE,,BUY,324.40,108.1,116,-7.9,...,37.0,2.25,37.0,50.7,1.0,0.101,-0.494,3.43,11.1,0.73
4,OEM-B.ST,SE,Sverige V8.1 Edge,POWER CORE,,BUY,151.20,102.9,140,-37.1,...,22.2,-6.06,18.5,63.1,5.3,0.075,-0.187,0.97,38.0,0.48
5,THULE.ST,SE,Sverige V8.1 Edge,POWER CORE,,BUY,235.80,62.1,128,-65.9,...,16.7,-13.69,0.0,57.4,0.6,0.147,-0.001,2.07,35.6,0.86
6,EQT.ST,SE,Sverige V8.1 Edge,POWER CORE,,BUY,305.40,57.3,100,-42.7,...,31.2,-6.00,12.5,51.5,-0.6,0.021,-0.454,1.45,20.3,0.99
7,ARJO-B.ST,SE,Sverige V8.1 Edge,POWER CORE,,BUY,24.92,44.6,110,-65.4,...,0.0,-12.60,0.0,50.3,0.7,0.008,0.006,2.24,15.8,0.65
8,VIT-B.ST,SE,Sverige V8.1 Edge,POWER CORE,,BUY,254.40,NaN,84,NaN,...,0.0,NaN,0.0,49.5,-3.0,0.093,-1.529,1.69,26.4,0.73
9,ABB.ST,SE,Sverige V8.1 Edge,EXHAUSTION WATCH,,TRIM,983.80,89.0,45,44.0,...,72.2,10.98,80.6,70.7,2.8,-0.904,-3.918,-3.12,44.3,1.27



EXIT / TRIM LIST — ALL MARKETS


,Ticker,Country,Model,OriginalSignal,ExitOverlay,Final,Price,RSI,RS20,RS60,...,Avg_10D,HitRate_10D,Avg_40D,HitRate_40D,KAMA_Dist_%,MACD_Accel,MACD_Accel_3D,DMI_Accel,ADX,Volume_Ratio
0,AAK.ST,SE,Sverige V8.1 Edge,EXHAUSTION WATCH,,TRIM,NaN,72.3,NaN,NaN,...,-2.88,0.0,-6.48,0.0,4.2,-0.090,-0.075,2.16,31.3,0.85
1,ABB.ST,SE,Sverige V8.1 Edge,EXHAUSTION WATCH,,TRIM,NaN,70.7,NaN,NaN,...,3.14,72.2,10.98,80.6,2.8,-0.904,-3.918,-3.12,44.3,1.27
2,SINCH.ST,SE,Sverige V8.1 Edge,EXHAUSTION WATCH,,TRIM,NaN,73.3,NaN,NaN,...,1.06,81.2,0.77,43.8,3.9,-0.123,-0.405,-1.79,50.8,0.72
3,TRUE-B.ST,SE,Sverige V8.1 Edge,EXHAUSTION WATCH,,TRIM,NaN,68.4,NaN,NaN,...,-8.50,0.0,-10.38,0.0,4.6,-0.023,-0.065,3.31,31.6,0.97
4,BALD-B.ST,SE,Sverige V8.1 Edge,AVOID,,EXIT,NaN,37.5,NaN,NaN,...,-2.19,12.5,-3.05,12.5,-0.5,-0.004,0.244,-4.18,24.5,1.35
5,BEIJ-B.ST,SE,Sverige V8.1 Edge,AVOID,,EXIT,NaN,38.6,NaN,NaN,...,1.44,100.0,-3.12,50.0,-3.7,0.210,0.262,5.00,30.0,1.41
6,BHG.ST,SE,Sverige V8.1 Edge,AVOID,,EXIT,NaN,46.5,NaN,NaN,...,-1.23,56.5,1.09,65.2,-2.8,0.006,-0.031,-0.19,15.9,0.65
7,HOLM-B.ST,SE,Sverige V8.1 Edge,AVOID,,EXIT,NaN,31.3,NaN,NaN,...,-2.42,25.0,-7.18,0.0,-2.6,0.147,-0.129,3.70,34.3,1.38
8,INDT.ST,SE,Sverige V8.1 Edge,AVOID,,EXIT,NaN,31.6,NaN,NaN,...,NaN,NaN,NaN,NaN,-3.9,0.227,0.891,-1.07,30.3,0.85
9,KINV-B.ST,SE,Sverige V8.1 Edge,AVOID,,EXIT,NaN,43.5,NaN,NaN,...,1.52,38.5,2.58,61.5,-1.7,0.012,-0.094,4.39,16.9,0.60



✅ Saved: POWERMODE_MASTER_2026-05-16.xlsx

NORWAY SUMMARY
Final
BUY           18
STRONG BUY     3
HOLD           2
Name: count, dtype: int64

GLOBAL SUMMARY
Final
WATCH         36
EXIT           7
BUY            6
TRIM           4
STRONG BUY     3
Name: count, dtype: int64

✅ Stored:
NORWAY_MASTER: (23, 16)
SWEDEN_MASTER: (56, 24)
USA_MASTER: (0, 0)
GLOBAL_MASTER: (56, 24)
EXIT_TRIM_LIST: (11, 33)


In [ ]:
from google.colab import files

files.download("POWERMODE_MASTER_2026-05-16.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>